In [1]:

import os
import sys
import sqlite3
import json
import subprocess
import importlib.util
from datetime import datetime

for package, module in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("plotly", "plotly"),
    ("ipywidgets", "ipywidgets"),
    ("IPython", "IPython"),
    ("tabulate", "tabulate")
]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", package
        ])

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

APP_NAME = "AFTERMATH AI"
DB_PATH = "aftermath_ai.db"
LAST_RESULTS = None

# ================================================================
# DATABASE
# ================================================================

def initialize_db():
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS decisions (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                title TEXT NOT NULL,
                created_at TEXT NOT NULL,
                scenarios TEXT NOT NULL,
                results TEXT NOT NULL,
                actual_scenario TEXT,
                actual_outcome REAL
            )
        """)

def save_decision(title, scenarios, summary):
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("""
            INSERT INTO decisions
            (title, created_at, scenarios, results)
            VALUES (?, ?, ?, ?)
        """, (
            title,
            datetime.now().isoformat(timespec="seconds"),
            json.dumps(scenarios),
            summary.to_json(orient="records")
        ))

def load_history():
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(
            "SELECT * FROM decisions ORDER BY id DESC", conn
        )

def record_actual(decision_id, scenario, outcome):
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("""
            UPDATE decisions
            SET actual_scenario = ?, actual_outcome = ?
            WHERE id = ?
        """, (scenario, float(outcome), int(decision_id)))

def initialize_app():
    initialize_db()

# ================================================================
# MONTE CARLO SIMULATION
# ================================================================

def simulate_scenario(scenario, iterations, seed):
    rng = np.random.default_rng(seed)

    cost_cv = scenario["cost_uncertainty"] / 100
    revenue_cv = scenario["revenue_uncertainty"] / 100

    cost_sigma = np.sqrt(np.log1p(cost_cv ** 2))
    revenue_sigma = np.sqrt(np.log1p(revenue_cv ** 2))

    cost_multiplier = rng.lognormal(
        mean=-0.5 * cost_sigma ** 2,
        sigma=cost_sigma,
        size=iterations
    )

    revenue_multiplier = rng.lognormal(
        mean=-0.5 * revenue_sigma ** 2,
        sigma=revenue_sigma,
        size=iterations
    )

    success = (
        rng.random(iterations)
        < scenario["success_probability"] / 100
    )

    costs = scenario["cost"] * cost_multiplier
    revenues = scenario["revenue"] * revenue_multiplier * success
    profits = revenues - costs

    q05, q25, q50, q75, q95 = np.percentile(
        profits, [5, 25, 50, 75, 95]
    )

    lower_tail = profits[profits <= q05]

    return {
        "name": scenario["name"],
        "input": scenario,
        "profits": profits,
        "Expected Profit": float(np.mean(profits)),
        "Median Profit": float(q50),
        "5th Percentile": float(q05),
        "25th Percentile": float(q25),
        "75th Percentile": float(q75),
        "95th Percentile": float(q95),
        "Probability of Loss (%)": float(np.mean(profits < 0) * 100),
        "Probability of Profit (%)": float(np.mean(profits > 0) * 100),
        "Expected Cost": float(np.mean(costs)),
        "Expected Revenue": float(np.mean(revenues)),
        "Mean Worst 5% Outcome": float(np.mean(lower_tail)),
        "Duration (months)": scenario["duration"]
    }

# ================================================================
# APPLICATION INTERFACE
# ================================================================

initialize_app()

header = widgets.HTML("""
<div style="padding:24px;border-radius:12px;
background:linear-gradient(120deg,#111827,#312e81);
color:white;margin-bottom:16px">
<h1 style="margin:0">AFTERMATH AI</h1>
<p>Predictive Decision Intelligence Platform</p>
<p>Simulate uncertainty, compare scenarios, and track actual outcomes.</p>
</div>
""")

title_input = widgets.Text(
    value="Should we launch a new product?",
    description="Decision:",
    layout=widgets.Layout(width="95%")
)

iterations_input = widgets.Dropdown(
    options=[1000, 5000, 10000, 25000, 50000],
    value=10000,
    description="Iterations:"
)

seed_input = widgets.IntText(
    value=42,
    description="Random seed:"
)

defaults = [
    ("Launch quickly", 30000, 60000, 60, 25, 35, 2),
    ("Test before launch", 45000, 85000, 75, 15, 20, 4),
    ("Small pilot", 20000, 45000, 70, 20, 25, 3)
]

scenario_inputs = []

for i, (name, cost, revenue, success, cost_unc, rev_unc, duration) in enumerate(defaults):
    name_widget = widgets.Text(
        value=name,
        description="Name:",
        layout=widgets.Layout(width="95%")
    )

    cost_widget = widgets.FloatText(
        value=float(cost),
        description="Cost:",
        layout=widgets.Layout(width="48%")
    )

    revenue_widget = widgets.FloatText(
        value=float(revenue),
        description="Revenue:",
        layout=widgets.Layout(width="48%")
    )

    success_widget = widgets.IntSlider(
        value=success,
        min=0,
        max=100,
        description="Success probability:",
        layout=widgets.Layout(width="95%")
    )

    cost_unc_widget = widgets.IntSlider(
        value=cost_unc,
        min=0,
        max=100,
        description="Cost uncertainty:",
        layout=widgets.Layout(width="95%")
    )

    revenue_unc_widget = widgets.IntSlider(
        value=rev_unc,
        min=0,
        max=100,
        description="Revenue uncertainty:",
        layout=widgets.Layout(width="95%")
    )

    duration_widget = widgets.IntSlider(
        value=duration,
        min=1,
        max=120,
        description="Duration (months):",
        layout=widgets.Layout(width="95%")
    )

    panel = widgets.VBox([
        widgets.HTML(f"<h3>Alternative {i + 1}</h3>"),
        name_widget,
        widgets.HBox([cost_widget, revenue_widget]),
        success_widget,
        cost_unc_widget,
        revenue_unc_widget,
        duration_widget
    ], layout=widgets.Layout(
        border="1px solid #888",
        padding="12px",
        margin="5px",
        width="32%"
    ))

    scenario_inputs.append({
        "panel": panel,
        "name": name_widget,
        "cost": cost_widget,
        "revenue": revenue_widget,
        "success": success_widget,
        "cost_unc": cost_unc_widget,
        "revenue_unc": revenue_unc_widget,
        "duration": duration_widget
    })

run_button = widgets.Button(
    description="Run Simulation",
    button_style="primary",
    layout=widgets.Layout(width="220px", height="42px")
)

save_button = widgets.Button(
    description="Save Decision",
    button_style="success",
    layout=widgets.Layout(width="180px", height="42px"),
    disabled=True
)

csv_button = widgets.Button(
    description="Download CSV",
    layout=widgets.Layout(width="180px", height="42px"),
    disabled=True
)

report_button = widgets.Button(
    description="Download Report",
    layout=widgets.Layout(width="180px", height="42px"),
    disabled=True
)

history_button = widgets.Button(
    description="Load Decision History",
    layout=widgets.Layout(width="220px", height="42px")
)

output = widgets.Output()
history_output = widgets.Output()

# ================================================================
# VALIDATION AND RESULTS
# ================================================================

def get_scenarios():
    scenarios = []

    for item in scenario_inputs:
        scenario = {
            "name": item["name"].value.strip(),
            "cost": float(item["cost"].value),
            "revenue": float(item["revenue"].value),
            "success_probability": int(item["success"].value),
            "cost_uncertainty": int(item["cost_unc"].value),
            "revenue_uncertainty": int(item["revenue_unc"].value),
            "duration": int(item["duration"].value)
        }

        if not scenario["name"]:
            raise ValueError("Every scenario needs a name.")

        if scenario["cost"] < 0 or scenario["revenue"] < 0:
            raise ValueError("Cost and revenue cannot be negative.")

        scenarios.append(scenario)

    names = [s["name"].lower() for s in scenarios]

    if len(names) != len(set(names)):
        raise ValueError("Scenario names must be unique.")

    return scenarios

def render_plot(fig):
    fig.update_layout(
        template="plotly_white",
        margin=dict(l=25, r=25, t=65, b=30),
        font=dict(family="Arial", size=12)
    )
    display(fig)

def run_simulation(_):
    global LAST_RESULTS

    with output:
        clear_output(wait=True)

        try:
            title = title_input.value.strip()

            if not title:
                raise ValueError("Enter a decision title.")

            scenarios = get_scenarios()
            iterations = int(iterations_input.value)
            seed = int(seed_input.value)

            results = [
                simulate_scenario(s, iterations, seed + i)
                for i, s in enumerate(scenarios)
            ]

            summary = pd.DataFrame([
                {
                    key: value
                    for key, value in result.items()
                    if key not in ("profits", "input")
                }
                for result in results
            ])

            LAST_RESULTS = {
                "title": title,
                "scenarios": scenarios,
                "results": results,
                "summary": summary,
                "iterations": iterations,
                "seed": seed
            }

            save_button.disabled = False
            csv_button.disabled = False
            report_button.disabled = False

            best = summary.loc[summary["Expected Profit"].idxmax()]
            lowest_risk = summary.loc[
                summary["Probability of Loss (%)"].idxmin()
            ]

            display(HTML("<h2>Simulation Results</h2>"))

            display(widgets.HBox([
                widgets.HTML(
                    f"<div style='padding:15px;border:1px solid #aaa;"
                    f"border-radius:10px;min-width:200px'>"
                    f"<b>Highest expected profit</b>"
                    f"<h3>{best['Expected Profit']:,.2f}</h3>"
                    f"{best['Scenario']}</div>"
                ),
                widgets.HTML(
                    f"<div style='padding:15px;border:1px solid #aaa;"
                    f"border-radius:10px;min-width:200px'>"
                    f"<b>Lowest probability of loss</b>"
                    f"<h3>{lowest_risk['Probability of Loss (%)']:.1f}%</h3>"
                    f"{lowest_risk['Scenario']}</div>"
                ),
                widgets.HTML(
                    f"<div style='padding:15px;border:1px solid #aaa;"
                    f"border-radius:10px;min-width:200px'>"
                    f"<b>Iterations per scenario</b>"
                    f"<h3>{iterations:,}</h3>Monte Carlo samples</div>"
                )
            ]))

            display(HTML("<h3>Scenario Comparison</h3>"))
            display(summary.round(2))

            fig = px.bar(
                summary,
                x="Scenario",
                y="Expected Profit",
                color="Scenario",
                title="Expected Profit by Scenario"
            )
            fig.add_hline(y=0, line_dash="dash")
            render_plot(fig)

            distribution = pd.concat([
                pd.DataFrame({
                    "Scenario": result["name"],
                    "Simulated Profit": result["profits"][
                        :min(10000, iterations)
                    ]
                })
                for result in results
            ], ignore_index=True)

            fig = px.violin(
                distribution,
                x="Scenario",
                y="Simulated Profit",
                color="Scenario",
                box=True,
                points=False,
                title="Distribution of Simulated Outcomes"
            )
            fig.add_hline(y=0, line_dash="dash")
            render_plot(fig)

            fig = px.bar(
                summary,
                x="Scenario",
                y="Mean Worst 5% Outcome",
                color="Scenario",
                title="Average Outcome in the Worst 5% of Simulations"
            )
            render_plot(fig)

            display(HTML("<h3>Decision Insights</h3>"))
            display(HTML(
                f"<p><b>Highest expected profit:</b> "
                f"{best['Scenario']}</p>"
                f"<p><b>Lowest probability of loss:</b> "
                f"{lowest_risk['Scenario']}</p>"
                "<p>Review downside risk and assumptions before deciding. "
                "The highest expected profit is not necessarily the "
                "most suitable alternative.</p>"
            ))

        except Exception as error:
            LAST_RESULTS = None
            save_button.disabled = True
            csv_button.disabled = True
            report_button.disabled = True
            display(HTML(
                f"<p style='color:#b91c1c'><b>Error:</b> "
                f"{str(error)}</p>"
            ))

# ================================================================
# SAVE AND EXPORT
# ================================================================

def save_current(_):
    if LAST_RESULTS is None:
        return

    save_decision(
        LAST_RESULTS["title"],
        LAST_RESULTS["scenarios"],
        LAST_RESULTS["summary"]
    )

    with output:
        display(HTML(
            "<p><b>Decision saved successfully.</b></p>"
        ))

def download_file(path):
    try:
        from google.colab import files
        files.download(path)
    except ImportError:
        display(HTML(
            f"<p>File saved to: {os.path.abspath(path)}</p>"
        ))

def download_csv(_):
    if LAST_RESULTS is None:
        return

    path = "aftermath_results.csv"
    LAST_RESULTS["summary"].to_csv(path, index=False)
    download_file(path)

def download_report(_):
    if LAST_RESULTS is None:
        return

    result = LAST_RESULTS

    report = [
        "# AFTERMATH AI Decision Report",
        "",
        f"Decision: {result['title']}",
        f"Generated: {datetime.now().isoformat(timespec='seconds')}",
        f"Iterations: {result['iterations']}",
        f"Random seed: {result['seed']}",
        "",
        "## Scenario Results",
        result["summary"].to_markdown(index=False),
        "",
        "## Limitations",
        "- Results are simulated estimates, not observed outcomes.",
        "- Results depend on user-provided assumptions.",
        "- Historical calibration is not performed.",
        "- Correlated risks and time value of money are not modeled."
    ]

    path = "aftermath_report.md"

    with open(path, "w", encoding="utf-8") as file:
        file.write("\n".join(report))

    download_file(path)

# ================================================================
# DECISION HISTORY AND AUDIT
# ================================================================

def show_history(_):
    with history_output:
        clear_output(wait=True)

        history = load_history()

        if history.empty:
            display(HTML("<p>No saved decisions yet.</p>"))
            return

        display(history[[
            "id", "title", "created_at",
            "actual_scenario", "actual_outcome"
        ]])

        options = [
            (f"#{row['id']} — {row['title']}", int(row["id"]))
            for _, row in history.iterrows()
        ]

        decision_selector = widgets.Dropdown(
            options=options,
            description="Decision:"
        )

        scenario_selector = widgets.Dropdown(
            options=[],
            description="Actual scenario:"
        )

        actual_input = widgets.FloatText(
            value=0.0,
            description="Actual outcome:"
        )

        record_button = widgets.Button(
            description="Record Actual Outcome",
            button_style="success"
        )

        def update_scenarios(change=None):
            selected = history.loc[
                history["id"] == decision_selector.value
            ].iloc[0]

            saved_scenarios = json.loads(selected["scenarios"])
            scenario_selector.options = [
                scenario["name"] for scenario in saved_scenarios
            ]

        decision_selector.observe(
            update_scenarios, names="value"
        )
        update_scenarios()

        def record_outcome(_):
            record_actual(
                decision_selector.value,
                scenario_selector.value,
                actual_input.value
            )
            with history_output:
                display(HTML("<p>Actual outcome recorded.</p>"))

        record_button.on_click(record_outcome)

        display(widgets.VBox([
            decision_selector,
            scenario_selector,
            actual_input,
            record_button
        ]))

        audit_button = widgets.Button(
            description="Show Prediction Audit"
        )

        def show_audit(_):
            updated_history = load_history()
            records = []

            for _, row in updated_history.iterrows():
                if pd.isna(row["actual_outcome"]):
                    continue

                if not row["actual_scenario"]:
                    continue

                saved_results = json.loads(row["results"])
                matched = [
                    item for item in saved_results
                    if item["name"] == row["actual_scenario"]
                ]

                if not matched:
                    continue

                predicted = float(matched[0]["Expected Profit"])
                actual = float(row["actual_outcome"])

                records.append({
                    "Decision": row["title"],
                    "Scenario": row["actual_scenario"],
                    "Expected Profit": predicted,
                    "Actual Outcome": actual,
                    "Absolute Error": abs(predicted - actual)
                })

            if records:
                audit = pd.DataFrame(records)
                display(audit)
                display(HTML(
                    f"<h4>Mean absolute error: "
                    f"{audit['Absolute Error'].mean():,.2f}</h4>"
                ))
            else:
                display(HTML(
                    "<p>No actual outcomes are available for auditing.</p>"
                ))

        audit_button.on_click(show_audit)
        display(audit_button)

# ================================================================
# LAUNCH APPLICATION
# ================================================================

run_button.on_click(run_simulation)
save_button.on_click(save_current)
csv_button.on_click(download_csv)
report_button.on_click(download_report)
history_button.on_click(show_history)

display(header)
display(HTML("<h3>1. Define Your Decision</h3>"))
display(title_input)
display(widgets.HBox([iterations_input, seed_input]))

display(HTML("<h3>2. Configure Alternatives</h3>"))
display(widgets.HBox(
    [item["panel"] for item in scenario_inputs],
    layout=widgets.Layout(width="100%", align_items="stretch")
))

display(widgets.HBox([run_button, save_button]))
display(widgets.HBox([csv_button, report_button]))
display(output)

display(HTML("<hr><h2>Decision History and Prediction Audit</h2>"))
display(history_button)
display(history_output)

display(HTML("""
<hr>
<p><b>AFTERMATH AI</b> is a decision simulation prototype.
It estimates outcomes using Monte Carlo sampling and user-defined
assumptions. It does not guarantee future results or replace
professional financial or business advice.</p>
"""))


HTML(value='\n<div style="padding:24px;border-radius:12px;\nbackground:linear-gradient(120deg,#111827,#312e81)…

Text(value='Should we launch a new product?', description='Decision:', layout=Layout(width='95%'))

Output()

Button(description='Load Decision History', layout=Layout(height='42px', width='220px'), style=ButtonStyle())

Output()